In [1]:
!pip install --upgrade langchain langchain-community langchain-text-splitters chromadb sentence-transformers pypdf google-generativeai langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.1/132.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.3/554.3 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/

In [3]:
from google.colab import files
uploaded = files.upload()
# Click the 'Choose Files' button that appears below to upload your documents.

Saving new resume.pdf to new resume.pdf


In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader, TextLoader
# Fixed import for text splitter to match the latest LangChain updates
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAI

# ==========================================
# STEP 1: Load Uploaded Documents
# ==========================================
document_paths = list(uploaded.keys())

if not document_paths:
    print("❌ Error: Please run Cell 2 first and upload your documents!")
else:
    all_documents = []
    for path in document_paths:
        if path.endswith(".pdf"):
            loader = PyPDFLoader(path)
        elif path.endswith(".txt"):
            loader = TextLoader(path)
        else:
            print(f"Skipping unsupported file type: {path}")
            continue
        all_documents.extend(loader.load())

    print(f"\n✅ Successfully loaded {len(all_documents)} document(s).")

    # Split documents into small chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = text_splitter.split_documents(all_documents)
    print(f"✂️ Split documents into {len(chunks)} text chunks.")

    # ==========================================
    # STEP 2: Vector Embeddings & ChromaDB Setup
    # ==========================================
    print("⏳ Generating embeddings and setting up Vector Store...")
    embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = Chroma.from_documents(documents=chunks, embedding=embedding_model, persist_directory="./chroma_db")
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
    print("📦 Chroma Vector Database is successfully ready!")

    # ==========================================
    # STEP 3: Configure Free Google Gemini LLM
    # ==========================================
    # ⚠️ Replace with your Gemini API key from Google AI Studio
    os.environ["GOOGLE_API_KEY"] = "ENTER YOUR API"
    llm = GoogleGenerativeAI(model="gemini-2.5-flash")

    # ==========================================
    # STEP 4: Define RAG Retrieval & Prompt Logic
    # ==========================================
    def ask_my_bot(query):
        retrieved_chunks = retriever.invoke(query)
        context = "\n\n".join([chunk.page_content for chunk in retrieved_chunks])

        prompt = f"""Answer the question based only on the context below.
If the answer isn't in the context, say "I don't have enough information to answer that."

Context:
{context}

Question: {query}

Answer:"""

        response = llm.invoke(prompt)
        return response

    # ==========================================
    # STEP 5: Interactive Chat Loop
    # ==========================================
    print("\n==========================================")
    print("🤖 AI Data Bot is Active! (Type 'exit' to stop)")
    print("==========================================\n")

    while True:
        user_query = input("You: ")
        if user_query.lower() == 'exit':
            print("Bot: Goodbye! Have a great day ahead.")
            break
        if user_query.strip() == "":
            continue

        print("⏳ Searching local vector database and generating answer...")
        bot_answer = ask_my_bot(user_query)
        print(f"\n🤖 BOT ANSWER:\n{bot_answer}")
        print("------------------------------------------\n")


✅ Successfully loaded 2 document(s).
✂️ Split documents into 14 text chunks.
⏳ Generating embeddings and setting up Vector Store...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

📦 Chroma Vector Database is successfully ready!

🤖 AI Data Bot is Active! (Type 'exit' to stop)

You: summarise
⏳ Searching local vector database and generating answer...

🤖 BOT ANSWER:
SREYALAKSHMI M K is an M.Sc. Applied Statistics & Data Analytics student with a CGPA of 8.92 and no active backlogs, based in Kannur, Kerala, India. She has hands-on experience in Power BI, Tableau, Excel, Python, and SQL, along with strong analytical and problem-solving skills. Her experience includes data analysis, KPI reporting, audit analytics, business process analysis, exception identification, and insight generation. She can be contacted via sreyalakshmimk3@gmail.com, +91 81297 53081, or her LinkedIn profile.
------------------------------------------

You: What are the core technical skills mentioned in this profile?  Based on this document, write a 3-sentence professional summary for a LinkedIn bio.
⏳ Searching local vector database and generating answer...

🤖 BOT ANSWER:
The core technical ski